# NPR-DeepfakeDetection — chạy đầy đủ bằng **Run All**

**Mục tiêu:** Nạp mô hình pretrained `NPR.pth`, phân loại ảnh Real/Fake, trực quan hóa 3 kênh NPR và lưu kết quả dự đoán nhiều ảnh. **Không training, không download dataset.**

### Cách dùng
1. Trong VS Code, chọn kernel `D:\XLA\.venv\Scripts\python.exe`.
2. Đảm bảo ảnh `D:\XLA\NPR-DeepfakeDetection\input_images\test.jpg` vẫn còn.
3. Nhấn **Run All**. Hình và kết quả CSV sẽ được lưu ở `NPR-DeepfakeDetection/outputs/`.
4. Muốn kiểm tra nhiều ảnh, thêm ảnh JPG/PNG vào thư mục `input_images`. Notebook dự đoán tối đa `MAX_IMAGES` ảnh mỗi lượt.

⚠️ `fake_score` là điểm **sigmoid chưa được hiệu chuẩn**. Dự đoán sai vẫn có thể xảy ra. Heatmap NPR **không phải** bản đồ vùng giả. Đánh giá Accuracy/AP chỉ có ý nghĩa trên ảnh **được gán nhãn đúng**.

## Cell 1 — Kiểm tra môi trường, đường dẫn và cấu hình

In [ ]:
from pathlib import Path
import sys, platform, json, csv, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image, ImageOps
from torchvision import transforms

# CẤU HÌNH: chỉnh tại đây nếu bạn đổi tên ảnh / thư mục
ROOT = Path(r"D:\XLA\NPR-DeepfakeDetection")
IMAGE_PATH = ROOT / "input_images" / "test.jpg"
CHECKPOINT_PATH = ROOT / "NPR.pth"
INPUT_DIR = ROOT / "input_images"
OUTPUT_DIR = ROOT / "outputs"
MAX_IMAGES = 50  # Không quét hàng nghìn ảnh trên CPU trong lần chạy đầu
THRESHOLD = 0.5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert ROOT.is_dir(), f"Không thấy thư mục dự án: {ROOT}"
assert CHECKPOINT_PATH.is_file(), f"Không thấy checkpoint: {CHECKPOINT_PATH}"
assert IMAGE_PATH.is_file(), f"Chưa có ảnh test: {IMAGE_PATH}"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Python:", sys.version.split()[0])
print("Python executable:", sys.executable)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Project:", ROOT)
print("Checkpoint MiB:", round(CHECKPOINT_PATH.stat().st_size / (1024**2), 2))
print("Test image:", IMAGE_PATH)
print("Output dir:", OUTPUT_DIR)

## Cell 2 — Nạp checkpoint vào **đúng mạng NPR của tác giả**

Repo này đặt tên hàm khởi tạo là `resnet50`, nhưng lớp mạng được tác giả tùy biến và chỉ giữ một số tầng của ResNet. **Không thay bằng `torchvision.models.resnet50`.**

In [ ]:
from networks.resnet import resnet50

ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
assert isinstance(ckpt, dict) and "model" in ckpt, "Checkpoint không có khóa 'model' như mong đợi"
state = {k.removeprefix("module."): v for k, v in ckpt["model"].items()}
model = resnet50(num_classes=1)
model.load_state_dict(state, strict=True)
model.to(DEVICE)
model.eval()

print("MODEL LOADED SUCCESSFULLY")
print("Số tensor/trạng thái trong checkpoint:", len(state))
print("Số tham số mô hình:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Số tham số có thể học:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Cell 3 — Đọc ảnh và chuẩn hóa giống cấu hình test ForenSynths

Cấu hình tác giả: resize về **256×256**, không crop, không flip, `ToTensor()`, `Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])`.

Đây là **pipeline ảnh đơn tương ứng với thiết lập resize của ForenSynths/DiffusionForensics**, không phải pipeline `no_resize=True` của mọi benchmark. Ảnh từ điện thoại có thể chứa thông tin định hướng EXIF nên khi đọc ta hiệu chỉnh định hướng trước.

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

with Image.open(IMAGE_PATH) as im:
    image = ImageOps.exif_transpose(im).convert("RGB")
x = preprocess(image).unsqueeze(0).to(DEVICE)

print("Original size (width, height):", image.size)
print("Tensor shape [batch, channel, height, width]:", tuple(x.shape))
fig, axes = plt.subplots(1,2,figsize=(10,4))
axes[0].imshow(image)
axes[0].set_title("Original image")
axes[1].imshow(image.resize((256,256), resample=Image.Resampling.BILINEAR))
axes[1].set_title("Resized input (256×256)")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

## Cell 4 — Trực quan hóa NPR thật từ hàm của mạng

Trong `networks/resnet.py`, tác giả tính `NPR = x - self.interpolate(x, 0.5)` **ngay trong `forward`**, sau đó truyền `NPR * (2/3)` vào lớp conv đầu. Ở đây ta chỉ xem NPR **trước** nhân hệ số 2/3, không làm biến đổi đầu vào lần thứ hai khi dự đoán.

Màu đỏ/xanh trên heatmap là **chênh lệch giá trị** theo thang `seismic`, không thể hiện xác suất ảnh giả.

In [ ]:
with torch.inference_mode():
    # Gọi đúng hàm interpolate của mô hình, không mô phỏng bằng thuật toán khác
    reconstructed = model.interpolate(x, 0.5)
    npr = x - reconstructed

npr_np = npr[0].detach().cpu().permute(1,2,0).numpy()
# Cùng thang màu cho R, G, B; lấy phân vị 99% tránh outlier làm hình mất chi tiết
scale = max(float(np.percentile(np.abs(npr_np), 99)), 1e-8)
channels = ["Red", "Green", "Blue"]
fig, axes = plt.subplots(1,4,figsize=(17,4.5))
axes[0].imshow(image)
axes[0].set_title("Original")
axes[0].axis("off")
for i,name in enumerate(channels):
    axes[i+1].imshow(npr_np[:,:,i], cmap="seismic", vmin=-scale, vmax=scale)
    axes[i+1].set_title("NPR – " + name)
    axes[i+1].axis("off")
fig.suptitle("Neighboring Pixel Relationships (NPR)", fontsize=14)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "npr_rgb.png", dpi=160, bbox_inches="tight")
plt.show()
print("NPR tensor shape:", tuple(npr.shape))
print("Saved:", OUTPUT_DIR / "npr_rgb.png")

## Cell 5 — Chạy inference **một ảnh**

`model(x)` tự tính NPR nội bộ rồi đưa vào CNN. Không nạp `npr` đã tính ở Cell 4 cho `model`, vì như vậy sẽ làm NPR hai lần.

In [ ]:
with torch.inference_mode():
    logit = model(x)
    fake_score = float(torch.sigmoid(logit).squeeze().item())

prediction = "FAKE" if fake_score > THRESHOLD else "REAL"
print("Logit:", float(logit.item()))
print("Fake score (scientific):", f"{fake_score:.6e}")
print("Prediction:", prediction)
print("Chú ý: score gần 0/1 không phải là xác suất đã hiệu chuẩn")

fig, ax = plt.subplots(figsize=(6,5))
ax.imshow(image)
ax.set_title(f"NPR: {prediction} | fake_score={fake_score:.4g}")
ax.axis("off")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "single_prediction.png", dpi=160, bbox_inches="tight")
plt.show()

## Cell 6 — Hàm dùng lại `predict(image)`

Nhận ảnh dạng **đường dẫn** hoặc **PIL Image**, trả dictionary. Đây là API tiện dụng cho các thành viên khác; hàm đã chuẩn hóa đầu vào giống Cell 3 và không train mô hình.

In [ ]:
def predict(img, model=model, device=DEVICE):
    if isinstance(img, (str, Path)):
        with Image.open(img) as loaded:
            img_rgb = ImageOps.exif_transpose(loaded).convert("RGB")
    elif isinstance(img, Image.Image):
        img_rgb = ImageOps.exif_transpose(img).convert("RGB")
    else:
        raise TypeError("img phải là đường dẫn hoặc PIL.Image")
    tensor = preprocess(img_rgb).unsqueeze(0).to(device)
    with torch.inference_mode():
        raw = float(model(tensor).squeeze().item())
    score = float(torch.sigmoid(torch.tensor(raw)).item())
    return {"label": "FAKE" if score > THRESHOLD else "REAL",
            "fake_score": score, "logit": raw}

example_result = predict(IMAGE_PATH)
print("API predict(test.jpg):", example_result)

## Cell 7 — Dự đoán nhiều ảnh, xuất CSV

Quét `input_images/` (cả các thư mục con `real/`, `fake/` nếu có), nhưng giới hạn ở `MAX_IMAGES` để tránh chạy quá lâu trên CPU. Một ảnh lỗi sẽ được ghi lại chứ không làm ngừng toàn bộ notebook.

In [ ]:
extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
files = sorted(p for p in INPUT_DIR.rglob("*") if p.is_file() and p.suffix.lower() in extensions)
files = files[:MAX_IMAGES]
rows = []
failures = []
print("Images selected:", len(files), "/ max", MAX_IMAGES)
start = time.perf_counter()

for idx, path in enumerate(files, start=1):
    try:
        r = predict(path)
        # Chỉ chấp nhận nhãn thật khi ảnh nằm trực tiếp trong thư mục real hoặc fake
        true_label = path.parent.name.lower() if path.parent.name.lower() in {"real", "fake"} else ""
        rows.append({"file": str(path.relative_to(INPUT_DIR)),
                     "true_label": true_label,
                     "prediction": r["label"],
                     "fake_score": r["fake_score"],
                     "logit": r["logit"]})
    except Exception as exc:
        failures.append({"file": str(path), "error": str(exc)})
    if idx % 10 == 0 or idx == len(files):
        print(f"Processed {idx}/{len(files)}")

csv_path = OUTPUT_DIR / "predictions.csv"
with csv_path.open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=["file", "true_label", "prediction", "fake_score", "logit"])
    writer.writeheader()
    writer.writerows(rows)
print("Predicted:", len(rows), "| Failed:", len(failures), "| Seconds:", round(time.perf_counter()-start,2))
print("Saved CSV:", csv_path)
for row in rows[:10]:
    print(f"{row['file']} | {row['prediction']} | score={row['fake_score']:.4g}")
if failures:
    print("Lỗi đầu tiên:", failures[0])

## Cell 8 — Tổng hợp kết quả và đánh giá **nếu có nhãn**

Nếu ảnh nằm trong `input_images/real/` và `input_images/fake/`, notebook sẽ tính Accuracy, Precision, Recall, F1 và AP **khi đủ cả hai lớp**. Nếu chỉ có `test.jpg` không gán nhãn, notebook chỉ báo số lượng dự đoán; **không tự nhận Accuracy**.

Đây là đánh giá nhỏ để kiểm tra pipeline, **không phải tái hiện benchmark 28 nguồn của paper**.

In [ ]:
from collections import Counter
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score, confusion_matrix

counts = Counter(r["prediction"] for r in rows)
print("Số ảnh dự đoán:", dict(counts))
labeled = [r for r in rows if r["true_label"] in {"real", "fake"}]
classes_present = {r["true_label"] for r in labeled}
if labeled and classes_present == {"real", "fake"}:
    y_true = np.array([1 if r["true_label"] == "fake" else 0 for r in labeled])
    scores = np.array([r["fake_score"] for r in labeled])
    y_pred = (scores > THRESHOLD).astype(int)
    metrics = {
        "count": len(labeled), "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "average_precision": average_precision_score(y_true, scores)
    }
    print("Đánh giá trên tập con gán nhãn (không phải benchmark paper):")
    for k,v in metrics.items(): print(k, ":", round(float(v), 4))
    print("Confusion matrix [REAL=0, FAKE=1]:\n", confusion_matrix(y_true, y_pred, labels=[0,1]))
    (OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
else:
    print("Chưa có đủ hai lớp nhãn REAL và FAKE. Bỏ qua Accuracy/F1/AP.")

run_info = {"python": sys.version.split()[0], "torch": torch.__version__,
            "device": str(DEVICE), "checkpoint": str(CHECKPOINT_PATH),
            "image": str(IMAGE_PATH), "threshold": THRESHOLD,
            "resize": [256,256], "max_images": MAX_IMAGES,
            "images_predicted": len(rows), "failures": failures}
(OUTPUT_DIR / "run_info.json").write_text(json.dumps(run_info, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved run info:", OUTPUT_DIR / "run_info.json")

## Bạn vừa hoàn thành những gì?

- ✅ Nạp `NPR.pth` và chạy **đúng kiến trúc** `networks/resnet.py` của tác giả.
- ✅ Xử lý ảnh RGB và thấy NPR ở ba kênh.
- ✅ Dự đoán ảnh đơn hoặc nhiều ảnh bằng CPU/GPU tự động.
- ✅ Lưu `outputs/npr_rgb.png`, `single_prediction.png`, `predictions.csv`, `run_info.json`.
- ✅ Chỉ tính metric khi có nhãn; **không train**, không đụng tới các file Python gốc.

**Nguồn đối chiếu:**
- [networks/resnet.py](https://github.com/chuangchuangtan/NPR-DeepfakeDetection/blob/main/networks/resnet.py)
- [data/datasets.py](https://github.com/chuangchuangtan/NPR-DeepfakeDetection/blob/main/data/datasets.py)
- [validate.py](https://github.com/chuangchuangtan/NPR-DeepfakeDetection/blob/main/validate.py)

**Lưu ý phương pháp:** hiện bạn mới thực hiện inference và một kiểm thử nhỏ; bước **tái hiện bảng Accuracy/AP trong paper** cần dataset và giao thức đánh giá chuẩn, nên phải làm riêng sau.